In [1]:
import pandas as pd
import sqlite3 as sq
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import requests
pd.set_option('display.max_rows', 1000); pd.set_option('display.max_columns', 1000); pd.set_option('display.width', 1000)
pd.options.mode.chained_assignment = None
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [2]:
data = pd.read_excel(r'G:\DATA\REQUESTS_CLIENTS\GNRC\20240215_UrbanSim_Update\Model Outputs\Small_Area.xlsx', sheet_name = None)

In [8]:
for sheet_name, df in data.items():
    print(f"Sheet Name: {sheet_name}")
    print(df.head())

Sheet Name: SUB-IP-EST2023-POP-47
  table with row headers in column A and column headers in rows 3 through 4 (leading dots indicate sub-parts)                     Unnamed: 1                          Unnamed: 2  Unnamed: 3  Unnamed: 4  Unnamed: 5
0  Annual Estimates of the Resident Population fo...                                                                                     NaN                                 NaN         NaN         NaN         NaN
1                                    Geographic Area                                                           April 1, 2020\nEstimates Base  Population Estimate (as of July 1)         NaN         NaN         NaN
2                                                NaN                                                                                     NaN                                2020      2021.0      2022.0      2023.0
3                              Adams city, Tennessee                                                              

In [6]:
parcel_lookup = data['parcel lookup']

In [7]:
#NAME is Census Place
#NAMELSAD is County
#Name is name of UGB: urban growth boundary, PGA: planned growth area
parcel_lookup.head()

,OBJECTID *,Shape *,Join_Count,TARGET_FID,urbansim_r,X,Y,X_LP,Y_LP,NAME,NAMELSAD,Shape_Length,Name,Type
0,669347,Point,1,669346,15324,-87.013953,36.135439,-87.013953,36.135439,NaN,Cheatham,953969.243122,Peagram UGB,UGB
1,669350,Point,1,669349,15351,-87.011172,36.135244,-87.011172,36.135244,NaN,Cheatham,953969.243122,Peagram UGB,UGB
2,669352,Point,1,669351,15330,-87.013097,36.134054,-87.013097,36.134054,NaN,Cheatham,953969.243122,Peagram UGB,UGB
3,669353,Point,1,669352,15350,-87.009619,36.134734,-87.009619,36.134734,NaN,Cheatham,953969.243122,Peagram UGB,UGB
4,669354,Point,1,669353,15273,-87.012648,36.132675,-87.012648,36.132675,NaN,Cheatham,953969.243122,Peagram UGB,UGB


In [9]:
#parcel_lookup['Type'].unique()

array(['UGB', 'PGA', nan], dtype=object)

# Sheet Names

+ SUB-IP-EST2023-POP-47: 2023 vintage pop estimates for incorporated places in Tennessee, years 2020 through 2023  
+ parcel lookup: each parcel tagged to a place, county, ugb/pga/none, with size and centroids  
+ Run X - parcel forecast 2023: 

In [11]:


# Read the Parquet file into a DataFrame
df = pd.read_parquet('../data/urbansim/20250512-181347-47197701-dcuncivfhlt8g6je9nfa-sim-results.parquet')

# Print the DataFrame
df.head()

,year,geo_level,geo_level_id,indicator_id,indicator_value
0,2023,block,470831203003000,sum_total_households,9.0
1,2023,block,470831203003090,sum_total_households,0.0
2,2023,block,471190103013047,sum_total_households,0.0
3,2023,block,471690902001031,sum_total_households,32.0
4,2023,block,471190112003048,sum_total_households,1.0


In [12]:
df['year'].unique()

array([2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033,
       2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044,
       2045, 2046, 2047, 2048, 2049, 2050])

In [13]:
df['geo_level'].unique()

array(['block', 'tract', 'block_group', 'region', 'county', 'municipal'],
      dtype=object)

In [14]:
df['indicator_id'].unique()

array(['sum_total_households', 'household_population', 'sum_total_units',
       'sum_total_jobs', 'residential_price', 'density_households_acre',
       'household_population_density_acres', 'density_units_acre',
       'density_jobs_acre', 'rent_burden', 'severe_rent_burden',
       'percentage_rent_burden', 'percentage_severe_rent_burden',
       'rental_units', 'own_units', 'renters_vacancy_rate',
       'owners_vacancy_rate', 'residential_capacity',
       'employment_capacity', 'sector_0', 'sector_1', 'sector_2',
       'sector_3', 'sector_4', 'sector_5', 'sector_6', 'sector_7',
       'sector_8', 'ln_price_own_county_47037',
       'prop_parcel_area_in_wetlands_county_other',
       'ln_price_own_county_other', 'ln_acres_county_other',
       'prop_parcel_area_slope_gt14_03_degrees_county_47037',
       'prop_parcel_area_slope_gt8_53_degrees_county_other',
       'ln_unit_county_other', 'prop_lowinc_county_other',
       'ln_job_access_10_miles_county_47037',
       'prop_parcel

So I'm taking the sample download of parcel data from UrbanSim and using the parcel lookup to populate the SmallArea and the Summary_w_UGB. 